# Week 12 Beginner Notebook: Loading Data Into Atlas and Supabase

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lolusername/CST4714_DB_admin/blob/main/week_12/week12_atlas_supabase_dataset_demo.ipynb)

This notebook is a basic walkthrough.

You will learn how to:

1. get a small real dataset
2. look at the data before loading it anywhere
3. connect Python to a remote MongoDB Atlas database
4. insert records into Atlas
5. connect Python to a remote Supabase/Postgres database
6. insert records into Supabase/Postgres

The dataset is a small sample from CISA's Known Exploited Vulnerabilities catalog.

## Big idea

A remote database is not stored on your laptop.

Your Python notebook connects to it over the internet using:

- a database address
- a username
- a password or access key
- permission from the cloud database to accept your connection

For safety, do not write passwords directly into the notebook.
In Colab, use the Secrets panel when possible.

## Before you start

If you want to run the Atlas cells, you need:

- a MongoDB Atlas cluster
- a MongoDB database user
- your current Colab public IP address allowed in Atlas Network Access
- a MongoDB connection string saved as a Colab secret named `MONGODB_URI`

If you want to run the Supabase/Postgres cells, you need:

- a Supabase project
- a Postgres connection string saved as a Colab secret named `SUPABASE_DB_URL`

Important Atlas note: Colab runs on Google's servers, not on your laptop.
That means Atlas sees the Colab runtime's public IP address.
If Atlas Network Access does not allow that IP address, the connection can fail even if your username and password are correct.

The notebook still works as a teaching demo if you skip the cloud write cells.

In [ ]:
# Install the small set of packages used in this notebook.
# pandas reads CSV files.
# requests downloads JSON from a public URL.
# pymongo connects to MongoDB Atlas.
# certifi gives Python a current CA certificate bundle for TLS.
# sqlalchemy and psycopg connect to Supabase/Postgres.

%pip -q install pandas requests "pymongo[srv]" certifi sqlalchemy "psycopg[binary]"

In [ ]:
import os
from getpass import getpass

import pandas as pd
import requests

## Step 1: Download data from a simple public API

An API is a URL that returns data instead of a normal web page.

This CISA URL returns JSON data.
JSON looks like Python dictionaries and lists, so it is a good fit for MongoDB documents.

In [ ]:
api_url = "https://www.cisa.gov/sites/default/files/feeds/known_exploited_vulnerabilities.json"

response = requests.get(api_url)
response.raise_for_status()

data = response.json()

print(type(data))
print(data.keys())
print("Number of vulnerabilities:", data["count"])

In [ ]:
# The actual vulnerability records are inside the "vulnerabilities" list.

records_from_api = data["vulnerabilities"]

print(type(records_from_api))
print("Records downloaded:", len(records_from_api))
print("First record:")
records_from_api[0]

## Step 2: Load the sample CSV file

CSV means comma-separated values.

CSV is a common format for tabular data.
It is often a natural fit for Postgres/Supabase because it looks like a table.

This notebook reads the sample CSV from the course GitHub repo.

In [ ]:
csv_url = "https://raw.githubusercontent.com/lolusername/CST4714_DB_admin/main/week_12/sample_cisa_kev_vulnerabilities.csv"

csv_df = pd.read_csv(csv_url)

print("Rows and columns:", csv_df.shape)
csv_df.head()

In [ ]:
# Look at the column names before loading anything into a database.

for column in csv_df.columns:
    print(column)

## Step 3: Keep the demo small

The sample CSV has 200 rows.
For a class demo, we will only insert 25 rows.

Small demos are easier to inspect and easier to delete later.

In [ ]:
demo_df = csv_df.head(25).copy()

print("Demo rows:", len(demo_df))
demo_df[["cveID", "vendorProject", "product", "dateAdded"]].head()

## Step 4: Make simple MongoDB documents

MongoDB stores documents.
In Python, a MongoDB document looks like a dictionary.

We will rename a few fields to beginner-friendly snake_case names.

In [ ]:
mongo_documents = []

for row in demo_df.to_dict(orient="records"):
    document = {
        "cve_id": row["cveID"],
        "vendor_project": row["vendorProject"],
        "product": row["product"],
        "vulnerability_name": row["vulnerabilityName"],
        "date_added": row["dateAdded"],
        "required_action": row["requiredAction"],
        "known_ransomware_campaign_use": row["knownRansomwareCampaignUse"],
        "source": "CISA KEV sample for CST4714 Week 12",
    }
    mongo_documents.append(document)

print("Documents ready:", len(mongo_documents))
mongo_documents[0]

## Atlas network access check

Before connecting to Atlas, find the public IP address of this notebook runtime.

In MongoDB Atlas, go to:

```text
Security -> Network Access -> Add IP Address
```

For a short class demo, you can add the IP printed below.
If the Colab runtime restarts, the IP may change.


In [ ]:
# This is the public IP address Atlas sees for this Colab runtime.
# Add this IP address in Atlas Network Access before running the ping cell.

public_ip = requests.get("https://api.ipify.org", timeout=10).text
print("Colab public IP address:", public_ip)
print("Add this IP in MongoDB Atlas Network Access if Atlas is blocking the connection.")


## Step 5: Connect to MongoDB Atlas

This is the important remote database idea.

Python does not magically know where Atlas is.
It needs a connection string.

A MongoDB Atlas connection string usually starts with:

```text
mongodb+srv://
```

Do not paste your real connection string into a notebook you will submit or publish.
Use Colab Secrets if possible.

In [ ]:
# In Colab, put your Atlas connection string in a secret named MONGODB_URI.
# If you are not in Colab, this cell will ask you to paste it.
# The pasted value is not saved in the notebook file.

try:
    from google.colab import userdata
    mongodb_uri = userdata.get("MONGODB_URI")
except Exception:
    mongodb_uri = os.environ.get("MONGODB_URI")

if not mongodb_uri:
    mongodb_uri = getpass("Paste MongoDB Atlas connection string: ")

print("Connection string loaded:", bool(mongodb_uri))
print("Connection string starts correctly:", mongodb_uri.startswith("mongodb+srv://") or mongodb_uri.startswith("mongodb://"))

# Do not print the actual connection string because it contains your username/password.

In [ ]:
# This cell connects to Atlas and sends a ping.
# Ping is a simple test that proves the remote database answered.
#
# Important: do NOT use tlsInsecure=True for normal class use.
# Instead, use certifi so Python has a current trusted CA certificate bundle.

import certifi
from pymongo import MongoClient
from pymongo.errors import ServerSelectionTimeoutError

mongo_client = MongoClient(
    mongodb_uri,
    tlsCAFile=certifi.where(),
    serverSelectionTimeoutMS=15000,
)

try:
    mongo_client.admin.command("ping")
    print("Connected to MongoDB Atlas")
except ServerSelectionTimeoutError as error:
    print("Atlas connection failed before the database could respond.")
    print()
    print("Most common fixes:")
    print("1. In Atlas Network Access, add this Colab public IP:", public_ip)
    print("2. Make sure the database username and password in the URI are correct.")
    print("3. Make sure special characters in the password are URL-encoded.")
    print("4. Make sure the URI starts with mongodb+srv:// and points to the correct cluster.")
    print("5. Do not add tlsInsecure=True. Use certifi as shown in this cell.")
    print()
    print("Original error:")
    raise error

In [ ]:
# Choose a database and collection.
# If they do not exist yet, MongoDB creates them when data is inserted.

mongo_database = mongo_client["cst4714_week12"]
mongo_collection = mongo_database["cisa_kev_beginner_demo"]

# Remove old demo rows with the same source so rerunning the notebook stays clean.
mongo_collection.delete_many({"source": "CISA KEV sample for CST4714 Week 12"})

insert_result = mongo_collection.insert_many(mongo_documents)

print("Inserted documents:", len(insert_result.inserted_ids))
print("Collection count:", mongo_collection.count_documents({}))

In [ ]:
# Read a few documents back from Atlas.
# This proves the data is actually in the remote database.

atlas_results = list(mongo_collection.find({}, {"_id": 0}).limit(5))
pd.DataFrame(atlas_results)

## Step 6: Prepare rows for Supabase/Postgres

Postgres stores data in tables.

A table needs columns and data types.
For this beginner demo, we will create a simple table with text columns.

In [ ]:
postgres_rows = []

for row in demo_df.to_dict(orient="records"):
    postgres_rows.append({
        "cve_id": row["cveID"],
        "vendor_project": row["vendorProject"],
        "product": row["product"],
        "vulnerability_name": row["vulnerabilityName"],
        "date_added": row["dateAdded"],
        "required_action": row["requiredAction"],
        "known_ransomware_campaign_use": row["knownRansomwareCampaignUse"],
    })

print("Rows ready:", len(postgres_rows))
postgres_rows[0]

## Step 7: Connect to Supabase/Postgres

Supabase is managed Postgres.
That means we can connect to it with a normal Postgres connection string.

A Postgres connection string usually starts with:

```text
postgresql://
```

Save the connection string as a Colab secret named `SUPABASE_DB_URL`.

In [ ]:
try:
    from google.colab import userdata
    supabase_db_url = userdata.get("SUPABASE_DB_URL")
except Exception:
    supabase_db_url = os.environ.get("SUPABASE_DB_URL")

if not supabase_db_url:
    supabase_db_url = getpass("Paste Supabase/Postgres connection string: ")

# SQLAlchemy needs to know which Postgres driver to use.
if supabase_db_url.startswith("postgresql://"):
    supabase_db_url = supabase_db_url.replace("postgresql://", "postgresql+psycopg://", 1)

print("Connection string loaded:", bool(supabase_db_url))

In [ ]:
from sqlalchemy import create_engine, text

engine = create_engine(supabase_db_url)

create_table_sql = """
create table if not exists public.cisa_kev_beginner_demo (
    cve_id text primary key,
    vendor_project text,
    product text,
    vulnerability_name text,
    date_added text,
    required_action text,
    known_ransomware_campaign_use text
);
"""

with engine.begin() as connection:
    connection.execute(text(create_table_sql))

print("Connected to Supabase/Postgres and created table if needed")

In [ ]:
# Insert rows into Supabase/Postgres.
# We delete old demo rows first so the notebook can be rerun cleanly.

insert_sql = """
insert into public.cisa_kev_beginner_demo (
    cve_id,
    vendor_project,
    product,
    vulnerability_name,
    date_added,
    required_action,
    known_ransomware_campaign_use
)
values (
    :cve_id,
    :vendor_project,
    :product,
    :vulnerability_name,
    :date_added,
    :required_action,
    :known_ransomware_campaign_use
)
on conflict (cve_id) do update set
    vendor_project = excluded.vendor_project,
    product = excluded.product,
    vulnerability_name = excluded.vulnerability_name,
    date_added = excluded.date_added,
    required_action = excluded.required_action,
    known_ransomware_campaign_use = excluded.known_ransomware_campaign_use;
"""

with engine.begin() as connection:
    connection.execute(text(insert_sql), postgres_rows)
    count = connection.execute(text("select count(*) from public.cisa_kev_beginner_demo")).scalar_one()

print("Rows now in Supabase/Postgres table:", count)

In [ ]:
# Read a few rows back from Supabase/Postgres.
# This proves the data is in the remote database.

with engine.begin() as connection:
    result = connection.execute(
        text("select cve_id, vendor_project, product, date_added from public.cisa_kev_beginner_demo limit 5")
    )
    rows_back = result.mappings().all()

pd.DataFrame(rows_back)

## Step 8: Manual upload files

Sometimes you do not want to connect from Python yet.
You can still create files for manual upload.

- Use the CSV file for Supabase/Postgres import.
- Use the JSON file for MongoDB Atlas or MongoDB Compass import.

In [ ]:
# Save the demo data in both common formats.

demo_df.to_csv("week12_demo_for_supabase.csv", index=False)

pd.DataFrame(mongo_documents).to_json(
    "week12_demo_for_mongodb.json",
    orient="records",
    indent=2
)

print("Created week12_demo_for_supabase.csv")
print("Created week12_demo_for_mongodb.json")

## What you should understand now

You should be able to explain:

- an API can return JSON data from a URL
- a CSV file is tabular data
- MongoDB Atlas needs a MongoDB connection string
- Supabase/Postgres needs a Postgres connection string
- MongoDB stores documents in collections
- Postgres stores rows in tables
- remote databases require credentials and network access
- secrets should not be pasted into public notebooks